In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import nltk
from nltk.corpus import stopwords

In [ ]:
# Only needed once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [61]:
data = pd.read_csv('fakenews_sample.csv')
data.head()

,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary
0,0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,Sometimes the power of Christmas will make you...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Church Congregation Brings Gift to Waitresses ...,Ruth Harris,NaN,[''],NaN,NaN,NaN
1,1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,Zurich Times,NaN,[''],NaN,NaN,NaN
2,2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,Never Hike Alone: A Friday the 13th Fan Film U...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Never Hike Alone - A Friday the 13th Fan Film ...,NaN,NaN,[''],Never Hike Alone: A Friday the 13th Fan Film ...,NaN,NaN
3,3,768,awm.com,unreliable,http://awm.com/elusive-alien-of-the-sea-caught...,"When a rare shark was caught, scientists were ...",2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Elusive ‘Alien Of The Sea ‘ Caught By Scientis...,Alexander Smith,NaN,[''],NaN,NaN,NaN
4,4,791,bipartisanreport.com,clickbait,http://bipartisanreport.com/2018/01/21/trumps-...,Donald Trump has the unnerving ability to abil...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Trump’s Genius Poll Is Complete & The Results ...,Gloria Christie,NaN,[''],NaN,NaN,NaN


## Clean the Data

In [62]:
def clean_text(text):
    # 1. Håndter tomme værdier (vigtigt for det store datasæt)
    if pd.isna(text):
        return ""
    
    # 2. Gør alt til små bogstaver
    text = text.lower()

    # 3. Erstat URLs med et tag (bevarer information om at der var et link)
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)

    # 4. Erstat e-mails med et tag
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '<EMAIL>', text)

    # 5. Erstat datoer (f.eks. 13th) med et tag
    text = re.sub(r'[0-9]+[a-zA-Z]+', '<DATE>', text)

    # 6. Erstat resterende tal med et tag
    text = re.sub(r'[0-9]+', '<NUM>', text)
    
    # 7. Fjern specialtegn (behold kun bogstaver, tags og mellemrum)
    text = re.sub(r'[^a-z\s<>]', '', text)
    
    # 8. Fjern ekstra mellemrum, tabs og linjeskift
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

data["clean_content"] = data["content"].apply(clean_text)
data[["content", "clean_content"]].head()

,content,clean_content
0,Sometimes the power of Christmas will make you...,sometimes the power of christmas will make you...
1,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,awakening of <> strands of dna reconnecting wi...
2,Never Hike Alone: A Friday the 13th Fan Film U...,never hike alone a friday the <> fan film usa ...
3,"When a rare shark was caught, scientists were ...",when a rare shark was caught scientists were l...
4,Donald Trump has the unnerving ability to abil...,donald trump has the unnerving ability to abil...


In [65]:
data['tokens'] = data['clean_content'].apply(nltk.word_tokenize)
data[['clean_content', 'tokens']].head()

,clean_content,tokens
0,sometimes the power of christmas will make you...,"[sometimes, the, power, of, christmas, will, m..."
1,awakening of <> strands of dna reconnecting wi...,"[awakening, of, <, >, strands, of, dna, reconn..."
2,never hike alone a friday the <> fan film usa ...,"[never, hike, alone, a, friday, the, <, >, fan..."
3,when a rare shark was caught scientists were l...,"[when, a, rare, shark, was, caught, scientists..."
4,donald trump has the unnerving ability to abil...,"[donald, trump, has, the, unnerving, ability, ..."


In [73]:
# Store English stopwords into a variable
english_stopwords = set(stopwords.words('english'))
print(f"Number of stopwords: {len(english_stopwords)}")
print(list(english_stopwords)[:15]) # Display first 15 stopwords

data['filtered_tokens'] = data['tokens'].apply(lambda tokens: [word for word in tokens if word not in english_stopwords])
data[['tokens', 'filtered_tokens']].head()

Number of stopwords: 198
['all', "hadn't", 'or', 'very', 're', 'myself', 'theirs', 'their', 'there', 'as', 'its', "shan't", "we've", 'what', 'ourselves']


,tokens,filtered_tokens
0,"[sometimes, the, power, of, christmas, will, m...","[sometimes, power, christmas, make, wild, wond..."
1,"[awakening, of, <, >, strands, of, dna, reconn...","[awakening, <, >, strands, dna, reconnecting, ..."
2,"[never, hike, alone, a, friday, the, <, >, fan...","[never, hike, alone, friday, <, >, fan, film, ..."
3,"[when, a, rare, shark, was, caught, scientists...","[rare, shark, caught, scientists, left, blunde..."
4,"[donald, trump, has, the, unnerving, ability, ...","[donald, trump, unnerving, ability, ability, c..."


In [74]:
# Liste over alle ord i alle artikler
all_words_list = [word for tokens_list in data['tokens'] for word in tokens_list]

# Sæt af unikke ord (vocab) og beregn vocab size
vocab = set(all_words_list)

# Længden af vocab
vocab_size = len(vocab)

print(f"Antal tokens i alt (alle ord i alle artikler): {len(all_words_list)}")
print(f"Vocabulary size (unikke ord): {vocab_size}")
print(f"10 ord i the vocab (bare lige for at se): {list(vocab)[:10]}")

Antal tokens i alt (alle ord i alle artikler): 173381
Vocabulary size (unikke ord): 16471
10 ord i the vocab (bare lige for at se): ['backward', 'devolve', 'inflating', 'killing', 'probiotic', 'carter', 'encroach', 'machete', 'goods', 'eroding']


In [75]:
# Liste over alle ord i alle artikler
filtered_words_list = [word for tokens_list in data['filtered_tokens'] for word in tokens_list]

# Sæt af unikke ord (vocab) og beregn vocab size
vocab_filtered = set(filtered_words_list)

# Længden af vocab
filtered_vocab_size = len(vocab_filtered)

print(f"Antal tokens i alt (alle ord i alle artikler): {len(filtered_words_list)}")
print(f"Vocabulary size (unikke ord): {filtered_vocab_size}")
print(f"10 ord i the vocab (bare lige for at se): {list(vocab_filtered)[:10]}")

Antal tokens i alt (alle ord i alle artikler): 100005
Vocabulary size (unikke ord): 16339
10 ord i the vocab (bare lige for at se): ['backward', 'devolve', 'inflating', 'killing', 'probiotic', 'carter', 'encroach', 'machete', 'goods', 'eroding']
